# OTT Content Trend Analysis (Netflix catalog)

Dataset: Netflix Movies and TV Shows (Kaggle, Shivam Bansal). 8,807 titles, data till Sept 2021.

**Sawal jo mujhe solve karna hai:** Netflix ke catalog mein kya trend chal raha hai - genre, country, release year aur movie length ke hisaab se? Isse ek content team ko kya decision milega?

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sqlite3

sns.set_style("whitegrid")
df = pd.read_csv("netflix_titles.csv")
print(df.shape)
df.head()

In [ ]:
df.info()
print(df.isnull().sum())
print(df['type'].value_counts())

**Pehli nazar mein:** director mein sabse zyada missing hai (2634), country mein 831 aur cast mein 825. date_added mein sirf 10 missing hain. Director aur cast ka mujhe trend analysis mein kaam nahi hai, to unko chhod diya.

In [ ]:
# --- Cleaning ---
df['country'] = df['country'].fillna('Unknown')
df['rating'] = df['rating'].fillna('Not Rated')

# date_added mein extra space tha, strip karna pada
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')
df['year_added'] = df['date_added'].dt.year

# duration: movie ke liye "90 min", show ke liye "2 Seasons" - number alag nikala
df['duration_num'] = df['duration'].str.extract(r'(\d+)').astype(float)

# ek title ke kai country/genre hote hain, simple rakhne ke liye pehla wala le raha hoon
df['main_country'] = df['country'].str.split(',').str[0].str.strip()
df['main_genre'] = df['listed_in'].str.split(',').str[0].str.strip()

print("duplicates:", df.duplicated().sum())
df = df.drop_duplicates()
df[['title','type','main_country','main_genre','year_added','duration_num']].head()

**Cleaning notes:** duplicates 0 mile. Pehla country/genre lene ka matlab hai multi-country co-productions ek hi country mein count hongi - ye is analysis ki limitation hai, report mein bhi likhunga.

In [ ]:
# Chart 1: Movies vs TV Shows
sns.countplot(data=df, x='type', color='steelblue')
plt.title("Catalog mein Movies vs TV Shows")
plt.show()
print(df['type'].value_counts(normalize=True).round(3))

**Chart 1:** Catalog ka lagbhag 70% Movies hai (6131) aur 30% TV Shows (2676).

In [ ]:
# Chart 2: Release year ke hisaab se count (2010 ke baad)
yearly = (df[df['release_year'] >= 2010]
          .groupby(['release_year', 'type']).size().reset_index(name='count'))
sns.lineplot(data=yearly, x='release_year', y='count', hue='type', marker='o')
plt.title("Titles by release year (2010 onwards)")
plt.show()

**Chart 2:** Movies 2017-18 ke aas-paas peak par gayi aur uske baad girti gayi. TV Shows 2020 tak badhte rahe. Note: 2021 ka data adhura hai (dataset Sept 2021 tak ka hai), isliye wo girawat poori sach nahi.

In [ ]:
# Chart 3: Top 10 genres
top_genres = df['main_genre'].value_counts().head(10)
sns.barplot(x=top_genres.values, y=top_genres.index, color='steelblue')
plt.title("Top 10 genres (main genre)")
plt.xlabel("Titles")
plt.show()

**Chart 3:** Dramas (1600) aur Comedies (1210) clear top par hain. Teesre number par Action & Adventure aur Documentaries lagbhag barabar hain.

In [ ]:
# Chart 4: genre mix heatmap
top5 = df['main_genre'].value_counts().head(5).index
mix = df[df['main_genre'].isin(top5) & (df['release_year'] >= 2010)]
pivot = pd.crosstab(mix['release_year'], mix['main_genre'])
plt.figure(figsize=(9, 6))
sns.heatmap(pivot, cmap="YlOrRd", annot=True, fmt='d')
plt.title("Top 5 genres by release year")
plt.show()

**Chart 4:** Dramas aur Comedies dono 2018 mein peak par the (216 aur 138) aur uske baad girte gaye. Lekin International TV Shows 2018 ke 94 se badhke 2020 mein 115 ho gaye - jab baaki genres neeche ja rahe the tab ye upar gaya. 2021 adhura saal hai.

In [ ]:
# Chart 5: Top 10 countries
top_c = df[df['main_country'] != 'Unknown']['main_country'].value_counts().head(10)
sns.barplot(x=top_c.values, y=top_c.index, color='indianred')
plt.title("Top 10 content producing countries")
plt.xlabel("Titles")
plt.show()

**Chart 5:** US (3211) pehle number par hai aur India (1008) doosre par, UK teesre (628). India ka US ke baad hona mere liye surprise tha.

In [ ]:
# Chart 6: Movie duration trend
movies = df[df['type'] == 'Movie']
avg_dur = movies.groupby('release_year')['duration_num'].mean().reset_index()
avg_dur = avg_dur[avg_dur['release_year'] >= 2000]
sns.lineplot(data=avg_dur, x='release_year', y='duration_num', marker='o')
plt.title("Average movie length (minutes) by release year")
plt.show()

**Chart 6:** 2000-2002 mein average movie lagbhag 115-119 min ki thi, 2020 tak ye 92 min ke paas aa gayi. Yani movies chhoti ho rahi hain.

In [ ]:
# Chart 7 (bonus): ratings
sns.countplot(data=df, y='rating', order=df['rating'].value_counts().index, color='seagreen')
plt.title("Content ratings")
plt.show()

**Chart 7:** TV-MA (3207) aur TV-14 (2160) mil ke catalog ka lagbhag 61% hain - matlab zyadatar content adults/teens ke liye hai, kids ka share chhota hai.

## Plotly (interactive charts)
Ye charts Colab mein hover karke dekh sakte hain - exact number tooltip mein dikhta hai.

In [ ]:
import plotly.express as px

yearly_all = (df[df['release_year'] >= 2000]
              .groupby(['release_year', 'type']).size().reset_index(name='count'))
fig = px.line(yearly_all, x='release_year', y='count', color='type',
              title="Release trend by type (2000 onwards)")
fig.show()

**Plotly 1:** Hover karke dekha - Movies ka peak 2017 aur 2018 mein 767 hai aur TV Shows 2020 mein 436 par peak karte hain.

In [ ]:
country_counts = (df[df['main_country'] != 'Unknown']
                  .groupby('main_country').size().reset_index(name='titles'))
fig = px.choropleth(country_counts, locations='main_country', locationmode='country names',
                    color='titles', color_continuous_scale='Reds',
                    title="Titles by main country")
fig.show()

**Plotly 2:** Map par US sabse gehra hai, India uske baad. Europe aur Latin America mein zyadatar countries halki hain.

In [ ]:
genre_tree = (df.groupby(['type', 'main_genre']).size().reset_index(name='titles')
              .sort_values('titles', ascending=False).head(15))
fig = px.treemap(genre_tree, path=['type', 'main_genre'], values='titles',
                 title="Top genres inside Movies and TV Shows")
fig.show()

**Plotly 3:** Treemap mein dikhta hai ki Movies ke andar Dramas aur Comedies bade hain, aur TV Shows mein International TV Shows sabse bada hai.

## SQL part
Same data ko SQLite mein daal ke core numbers SQL se nikaal raha hoon.

In [ ]:
conn = sqlite3.connect("netflix.db")
df[['type','title','main_country','release_year','year_added','rating','main_genre','duration_num']] \
    .to_sql("titles", conn, index=False, if_exists="replace")

In [ ]:
q1 = """
SELECT main_genre, COUNT(*) AS total
FROM titles
GROUP BY main_genre
ORDER BY total DESC
LIMIT 10;
"""
pd.read_sql(q1, conn)

In [ ]:
q2 = """
SELECT year_added, type, COUNT(*) AS added
FROM titles
WHERE year_added IS NOT NULL
GROUP BY year_added, type
ORDER BY year_added;
"""
pd.read_sql(q2, conn)

In [ ]:
q3 = """
SELECT main_country, COUNT(*) AS total
FROM titles
WHERE main_country != 'Unknown'
GROUP BY main_country
HAVING COUNT(*) > 50
ORDER BY total DESC;
"""
pd.read_sql(q3, conn)

In [ ]:
q4 = """
SELECT release_year, ROUND(AVG(duration_num), 1) AS avg_minutes
FROM titles
WHERE type = 'Movie' AND release_year >= 2000
GROUP BY release_year
ORDER BY release_year;
"""
pd.read_sql(q4, conn)

## SQL joins
Ek chhoti lookup table banayi (country -> region) aur titles ke saath join kiya. Jo country table mein nahi hai wo LEFT JOIN se bhi aa jaye, isliye COALESCE se 'Other' likha.

In [ ]:
region_map = {
    'United States':'North America','Canada':'North America','Mexico':'Latin America',
    'Brazil':'Latin America','Argentina':'Latin America','Colombia':'Latin America',
    'United Kingdom':'Europe','France':'Europe','Spain':'Europe','Germany':'Europe',
    'Italy':'Europe','Turkey':'Europe','Ireland':'Europe',
    'India':'Asia','Japan':'Asia','South Korea':'Asia','China':'Asia','Thailand':'Asia',
    'Philippines':'Asia','Indonesia':'Asia','Hong Kong':'Asia','Taiwan':'Asia',
    'Egypt':'Middle East & Africa','Nigeria':'Middle East & Africa','South Africa':'Middle East & Africa',
    'Australia':'Oceania'}
regions = pd.DataFrame(list(region_map.items()), columns=['main_country', 'region'])
regions.to_sql("regions", conn, index=False, if_exists="replace")

In [ ]:
q5 = """
SELECT COALESCE(r.region, 'Other') AS region, t.type, COUNT(*) AS total
FROM titles t
LEFT JOIN regions r ON t.main_country = r.main_country
WHERE t.main_country != 'Unknown'
GROUP BY COALESCE(r.region, 'Other'), t.type
ORDER BY region, total DESC;
"""
pd.read_sql(q5, conn)

**Join result:** Har region mein Movies zyada hain. TV Shows ka hissa Europe mein ~33% hai (442 / 1330) aur Asia mein ~28% (562 / 1972). Middle East & Africa mein TV Shows bahut kam hain (29).

In [ ]:
q6 = """
SELECT main_country,
       COUNT(*) AS total,
       ROUND(100.0 * SUM(CASE WHEN rating IN ('TV-MA','R','NC-17') THEN 1 ELSE 0 END) / COUNT(*), 1) AS mature_pct
FROM titles
WHERE main_country != 'Unknown'
GROUP BY main_country
HAVING COUNT(*) > 100
ORDER BY mature_pct DESC;
"""
pd.read_sql(q6, conn)

**Note:** Dataset mein numeric rating (jaise 7.5/10) nahi hai, isliye 'average rating by country' nahi nikal sakta. Uske badle har country mein mature content (TV-MA/R/NC-17) ka % nikala. Spain (81%) aur Mexico (74%) sabse upar hain, US 47% par hai.

# Part 2: Netflix vs Amazon Prime
Veda ke task mein Netflix ke saath Prime bhi likha hai, to doosra dataset (amazon_prime_titles.csv, Kaggle) le ke dono platforms ko compare kiya. Is dataset mein bhi wahi 12 columns hain.

**Dhyan rakhne wali baatein:** Prime mein country sirf ~7% titles mein bhari hai aur date_added lagbhag sab khali hai. To country aur year_added ka comparison Prime ke liye reliable nahi hai, wo nahi kar raha. Is dataset mein numeric rating (IMDb jaisi) bhi nahi hai.

In [ ]:
prime = pd.read_csv("amazon_prime_titles.csv")
print(prime.shape)
print(prime.isnull().sum())
print(prime['type'].value_counts())

**Prime check:** 9,668 titles hain. country mein 8,996 aur date_added mein 9,513 missing hain, rating mein 337. Duplicates nahi mile.

In [ ]:
# Prime ki cleaning, Netflix jaisi hi
prime['rating'] = prime['rating'].fillna('Not Rated')
prime['duration_num'] = prime['duration'].str.extract(r'(\d+)').astype(float)
prime = prime.drop_duplicates()

# dono ko ek table mein jodna
nf = df[['type','title','release_year','rating','duration_num','listed_in']].copy()
nf['platform'] = 'Netflix'
pr = prime[['type','title','release_year','rating','duration_num','listed_in']].copy()
pr['platform'] = 'Prime'
both = pd.concat([nf, pr], ignore_index=True)
both['platform'].value_counts()

In [ ]:
# Chart 8: Movies vs TV Shows, dono platforms
split = pd.crosstab(both['platform'], both['type'], normalize='index') * 100
split.plot(kind='bar', stacked=True, color=['steelblue', 'darkorange'])
plt.title("Movies vs TV Shows share (%)")
plt.ylabel("%")
plt.xticks(rotation=0)
plt.show()
print(split.round(1))

**Chart 8:** Prime zyada movie-heavy hai: 81% movies aur 19% TV Shows. Netflix mein ye 70% / 30% hai.

In [ ]:
# Chart 9: release year ke hisaab se catalog ka share
recent = both[both['release_year'] >= 2000]
yr = recent.groupby(['release_year', 'platform']).size().reset_index(name='count')
sns.lineplot(data=yr, x='release_year', y='count', hue='platform', marker='o')
plt.title("Titles by release year (2000 onwards)")
plt.show()
for p_name, d in both.groupby('platform'):
    print(p_name, "2000 se pehle ka share:", round((d['release_year'] < 2000).mean() * 100, 1), "%")

**Chart 9:** Prime ka catalog purana bhi hai: 17.5% titles 2000 se pehle ke hain, Netflix mein sirf 6%. Netflix ke release years 2016-2020 ke aas-paas zyada ghane hain.

In [ ]:
# Chart 10: genre share (kitne % titles mein ye genre tag hai)
keywords = {'Drama':'Drama', 'Comedy':'Comed', 'Action':'Action', 'Documentary':'Docu',
            'Horror':'Horror', 'Romance':'Roman', 'Kids':'Kids|Children'}
rows = []
for plat, d in both.groupby('platform'):
    for g_name, pat in keywords.items():
        rows.append({'platform': plat, 'genre': g_name,
                     'pct': d['listed_in'].str.contains(pat).mean() * 100})
gshare = pd.DataFrame(rows)
sns.barplot(data=gshare, x='genre', y='pct', hue='platform')
plt.title("Genre share by platform (% of titles with that tag)")
plt.ylabel("% of titles")
plt.show()
gshare.pivot(index='genre', columns='platform', values='pct').round(1)

**Chart 10:** Drama dono mein lagbhag barabar hai (36% vs 38%). Netflix mein Comedy (30% vs 22%) aur Documentary (14% vs 10%) zyada hain. Prime mein Action (17% vs 12%) aur Horror (9% vs 5%) zyada hain. Ek title mein kai genre tag hote hain, isliye % jodke 100 nahi banta.

In [ ]:
# Chart 11: movie length trend
mv = both[(both['type'] == 'Movie') & (both['release_year'] >= 2000)]
dur = mv.groupby(['release_year', 'platform'])['duration_num'].mean().reset_index()
sns.lineplot(data=dur, x='release_year', y='duration_num', hue='platform', marker='o')
plt.title("Average movie length (min) by release year")
plt.show()
print(both[both['type'] == 'Movie'].groupby('platform')['duration_num'].mean().round(1))

**Chart 11:** Prime ki movies Netflix se chhoti hain: average 91 min vs 100 min. Dono mein length ghat rahi hai.

In [ ]:
# Chart 12: adult rated content
# rating systems alag hain: Netflix TV-MA/R/NC-17, Prime 18+/R/NC-17 ko adult maana
adult_ratings = {'Netflix': ['TV-MA', 'R', 'NC-17'], 'Prime': ['18+', 'R', 'NC-17']}
adult = {}
for plat, d in both.groupby('platform'):
    known = d[d['rating'] != 'Not Rated']
    adult[plat] = known['rating'].isin(adult_ratings[plat]).mean() * 100
plt.bar(list(adult.keys()), list(adult.values()), color=['steelblue', 'darkorange'])
plt.title("Adult rated titles (% of rated titles)")
plt.ylabel("%")
plt.show()
print({k: round(v, 1) for k, v in adult.items()})

**Chart 12:** Netflix mein 46% rated titles adult category ke hain, Prime mein 24%. Dhyan: dono ka rating system alag hai, to ye sirf approximate comparison hai.

In [ ]:
# Plotly: genre comparison (interactive)
fig = px.bar(gshare, x='genre', y='pct', color='platform', barmode='group',
             title="Genre share by platform")
fig.show()

In [ ]:
# Combined table SQL ke liye
both[['platform','type','title','release_year','rating','duration_num','listed_in']] \
    .to_sql("all_titles", conn, index=False, if_exists="replace")

q7 = """
SELECT platform, type, COUNT(*) AS total,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY platform), 1) AS pct_of_platform
FROM all_titles
GROUP BY platform, type;
"""
pd.read_sql(q7, conn)

In [ ]:
q8 = """
SELECT platform,
       ROUND(100.0 * SUM(CASE WHEN listed_in LIKE '%Comed%' THEN 1 ELSE 0 END) / COUNT(*), 1) AS comedy_pct,
       ROUND(100.0 * SUM(CASE WHEN listed_in LIKE '%Action%' THEN 1 ELSE 0 END) / COUNT(*), 1) AS action_pct,
       ROUND(100.0 * SUM(CASE WHEN listed_in LIKE '%Horror%' THEN 1 ELSE 0 END) / COUNT(*), 1) AS horror_pct
FROM all_titles
GROUP BY platform;
"""
pd.read_sql(q8, conn)

In [ ]:
q9 = """
SELECT platform, ROUND(AVG(duration_num), 1) AS avg_movie_minutes,
       SUM(CASE WHEN release_year < 2000 THEN 1 ELSE 0 END) AS pre_2000_titles
FROM all_titles
WHERE type = 'Movie'
GROUP BY platform;
"""
pd.read_sql(q9, conn)

**SQL result:** Prime ka movie average Netflix se chhota hai aur uska purana (2000 se pehle ka) catalog bhi bada hai.

In [ ]:
# Excel pivot ke liye clean file
df.to_csv("netflix_clean.csv", index=False)